In [0]:
%pip install ecmwf-api-client cdsapi xarray netcdf4 cfgrib

## ECMWF Data Access Setup

### For ECMWF MARS API (requires ECMWF member-state access):
1. Register at https://apps.ecmwf.int/registration/
2. Retrieve your API key from https://api.ecmwf.int/v1/key/
3. Create `~/.ecmwfapirc` file with:
```
{
    "url": "https://api.ecmwf.int/v1",
    "key": "your-api-key",
    "email": "your-email@example.com"
}
```

### For Climate Data Store (CDS) API (publicly accessible):
1. Register at https://cds.climate.copernicus.eu/
2. Get your API key from https://cds.climate.copernicus.eu/api-how-to
3. Create `~/.cdsapirc` file with:
```
url: https://cds.climate.copernicus.eu/api/v2
key: your-uid:your-api-key
```

In [0]:
import os

# Check the file the user mentioned
workspace_config = "/Workspace/Users/wei_jun_ang_from.tp@nea.gov.sg/aurora-databricks-pipeline/.ecmwfapirc"
home_config = os.path.expanduser("~/.ecmwfapirc")

print(f"Workspace config path: {workspace_config}")
print(f"Exists: {os.path.exists(workspace_config)}")

if os.path.exists(workspace_config):
    print("\nFile content:")
    with open(workspace_config, 'r') as f:
        content = f.read()
        print(content)
        
print(f"\n\nHome config path: {home_config}")
print(f"Exists: {os.path.exists(home_config)}")

In [0]:
import shutil
import os
from ecmwfapi import ECMWFService
from datetime import datetime, timedelta

output_file_sfc = "/dbfs/tmp/ec_data_sample_sfc.grib"

# Initialize MARS service (same as your working script)
server = ECMWFService("mars")

# Target parameters
target_valid_time = datetime(2024, 3, 2, 0)
area = "90/-180/-90/180"
grid = "0.1/0.1"
step = "00"

# Format date and time
date = target_valid_time.strftime("%Y-%m-%d")
time = target_valid_time.strftime("%H")

# Execute MARS request for surface variables
server.execute({
    "class": "od",
    "stream": "oper",
    "type": "an",
    "levtype": "sfc",
    "param": "167/165/166/151",  # 2t, 10u, 10v, msl
    "date": date,
    "time": time,
    "step": step,
    "grid": grid,
    "area": area,
    "expver": "1"
}, output_file_sfc)

print(f"✓ Data downloaded successfully to ec_data_sample_sfc.grib")

In [0]:
output_file_pl = "/dbfs/tmp/ec_data_sample_pl.grib"

server.execute({
    "class": "od",
    "stream": "oper",
    "type": "an",
    "levtype": "pl",
    "levelist": "1000/925/850/700/600/500/400/300/250/200/150/100/50",  
    "param": "130/131/132/133/129",  
    "date": date,
    "time": time,
    "step": step,
    "grid": grid,
    "area": area,
    "expver": "1"
}, output_file_pl)

print(f"✓ Pressure level data downloaded to {output_file_pl}")

In [0]:
import pygrib
import pandas as pd

# Check Surface Level GRIB
sfc_file = "/dbfs/tmp/ec_data_sample_sfc.grib"
grbs = pygrib.open(sfc_file)

print(f"=== Surface Level GRIB ({sfc_file}) ===")
print(f"Number of messages: {grbs.messages}\n")

for grb in grbs:
    print(f"Parameter: {grb.name} ({grb.shortName})")
    print(f"  Level: {grb.level} | Type of level: {grb.typeOfLevel}")
    print(f"  Date/Time: {grb.validDate}")
    print(f"  Grid: {grb.Ni}x{grb.Nj} | Values shape: {grb.values.shape}")
    print(f"  Min: {grb.values.min():.4f} | Max: {grb.values.max():.4f} | Mean: {grb.values.mean():.4f}")
    print()

grbs.close()

In [0]:
# Check Pressure Level GRIB
pl_file = "/dbfs/tmp/ec_data_sample_pl.grib"
grbs = pygrib.open(pl_file)

print(f"=== Pressure Level GRIB ({pl_file}) ===")
print(f"Number of messages: {grbs.messages}\n")

for grb in grbs:
    print(f"Parameter: {grb.name} ({grb.shortName})")
    print(f"  Level: {grb.level} hPa | Type of level: {grb.typeOfLevel}")
    print(f"  Date/Time: {grb.validDate}")
    print(f"  Grid: {grb.Ni}x{grb.Nj} | Values shape: {grb.values.shape}")
    print(f"  Min: {grb.values.min():.4f} | Max: {grb.values.max():.4f} | Mean: {grb.values.mean():.4f}")
    print()

grbs.close()